## Quick demo KGraphPy

In [2]:
import cim_plugin   # Required to register the cimxml parser and serializer in the rdflib library
from rdflib import Literal, URIRef
from rdflib.namespace import RDF, DCAT, DCTERMS
from cim_plugin.utilities import load_graphs_from_cimxml, load_graphs_from_trig
from cim_plugin.header import CIMMetadataHeader
from pathlib import Path


### Import graph from file
- use load_graphs_from_cimxml for one or more cimxml file(s)
- use load_graphs_from_trig for one trig file

In [3]:
input_file= Path.cwd().parents[1] / "Nordic44/instances/Grid/cimxml/Nordic44-HV_EQ.xml"

cims = load_graphs_from_cimxml([str(input_file)])

trig_file = Path.cwd().parents[1] / "Nordic44/instances/Grid/trig/Nordic44-HV_EQ.trig"

trigs = load_graphs_from_trig(str(trig_file))


### Handle the objects and identifiers
The load functions loads the graphs as a list of CIMProcessor objects. Each object can be extracted by indexing, and the graph identifier is stored as CIMProcessor.identifier.


In [4]:
g = cims[0]
t = trigs[0]

print("cimxml: ", g.identifier)
print("trig: ", t.identifier)

cimxml:  urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c
trig:  urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c


### Handling the graphs
The graph is stored in CIMProcess.graph. It can be handled with the usual rdflib functions and methods.

In [5]:
print("cimxml: ")
counter = 0
for s, p, o in g.graph:
    print(s, p, o)
    if isinstance(o, Literal):
        print(o.datatype)
    counter += 1
    if counter == 3:
        break

print()
print("trig: ")
counter = 0
for s, p, o in t.graph:
    print(s, p, o)
    if isinstance(o, Literal):
        print(o.datatype)
    counter += 1
    if counter == 3:
        break

cimxml: 
urn:uuid:9c68bc7f-428a-4616-9183-44d66f3ec531 http://iec.ch/TC57/CIM100#IdentifiedObject.mRID 9c68bc7f-428a-4616-9183-44d66f3ec531
None
urn:uuid:f1769f1c-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#IdentifiedObject.name EVLO
None
urn:uuid:f1769a4c-9aeb-11e5-91da-b8763fd99c5f http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://iec.ch/TC57/CIM100#ACLineSegment

trig: 
urn:uuid:ff3505b4-ebd4-164f-940a-339566d09ed7 https://cim.ucaiug.io/ns#Line.Region urn:uuid:f17695c3-9aeb-11e5-91da-b8763fd99c5f
urn:uuid:f1769e66-9aeb-11e5-91da-b8763fd99c5f https://cim.ucaiug.io/ns#OperationalLimit.OperationalLimitType urn:uuid:f1769a41-9aeb-11e5-91da-b8763fd99c5f
urn:uuid:2dd90420-bdfb-11e5-94fa-c8f73332c8f4 https://cim.ucaiug.io/ns#Terminal.ConnectivityNode urn:uuid:f1769693-9aeb-11e5-91da-b8763fd99c5f


### Header
CIMProcess.extract_header finds all the triples in the graph belonging to the metadata header, extracts them into another object and deletes them from the graph. The header can then be accessed through CIMProcess.graph.metadata_header.

In [6]:
g.extract_header()
t.extract_header()

print("cimxml: ")
for s, p, o in g.graph.metadata_header.triples:
    print(s, p, o)

print()
print("trig: ")
for s, p, o in t.graph.metadata_header.triples:
    print(s, p, o)

cimxml: 
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.description Equipment (EQ) part of the Nordic 44-bus synthetic test model developed by Statnett SF of the Nordic region.
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.modelingAuthoritySet http://www.Statnett.no/IGM/Nordic44_CGM
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.created 2025-02-14T12:00:00Z
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.profile http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.scenarioTime 2014-12-31T23:00:00Z
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.version 1
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://www.w3.org/1999/02/22-rdf-sy

### Replace header
You can replace one header with a new header using CIMProcess.replace_header. You can build the new header up triple by triple, or collect it from another file

In [7]:
new_header = CIMMetadataHeader.empty(URIRef("header_example")) # Creating a new empty header
new_header.graph.bind("ex", "example.org")  # Binding a new namespace
new_header.add_triple(RDF.type, DCAT.Dataset)   # The rdf.type triple is important to recognise the header
new_header.add_triple(DCTERMS.conformsTo, URIRef("http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0"))
new_header.add_triple(URIRef("example.org/predicate"), Literal("example object"))   # Adding a triple
new_header.profile = "http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0"   # Attach profile
t.replace_header(new_header)

print("trig: ")
for s, p, o in t.graph.metadata_header.triples:
    print(s, p, o)

trig: 
header_example http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/ns/dcat#Dataset
header_example http://purl.org/dc/terms/conformsTo http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0
header_example example.org/predicate example object


### Validate header
You can check if the header follows the cim standard for dcat:Dataset headers. The validation will correct where automatic correction is possible. Logs are emitted for every correction as well as for errors that must be fixed manually.
Trig and cimxml has slightly different standards, so format must be given. Default is "cimxml". 

In [8]:
g.validate_header()
t.validate_header(format="trig")

Validation for MD.FullModel header is not implemented yet. No validation performed for this header type.
Missing required http://purl.org/dc/terms/issued triple. Creating dummy triple without date.
Missing required http://www.w3.org/ns/dcat#endDate triple. Creating dummy triple without date.
Missing required http://www.w3.org/ns/dcat#startDate triple. Creating dummy triple without date.
Missing required rdf:type rdfg:Graph triple for Trig header. Adding it.


### Namespaces
Namespaces can be changed using CIMProcess.update_namespace. Though rdflibs graph.bind allows binding of new namespaces, it does not update the triples. This method does both, but can only be used on namespaces that already exists.

In [9]:
t.update_namespace("cim", g.graph.namespace_manager.store.namespace("cim")) # Updating the trig graph with the cim namespace from the cimxml graph.

print("trig: ")
counter = 0
for s, p, o in t.graph:
    print(s, p, o)
    counter += 1
    if counter == 3:
        break

trig: 
urn:uuid:9c68bc7f-428a-4616-9183-44d66f3ec531 http://iec.ch/TC57/CIM100#IdentifiedObject.mRID 9c68bc7f-428a-4616-9183-44d66f3ec531
urn:uuid:f1769f1c-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#IdentifiedObject.name EVLO
urn:uuid:f1769a4c-9aeb-11e5-91da-b8763fd99c5f http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://iec.ch/TC57/CIM100#ACLineSegment


### Serialize to cimxml
To serialize to file the rdflib graph.serialize method can be used. For cimxml, the header must be seperate from the graph (into CIMProcessor.graph.metadata_header). If it is not, the header will be serialized as regular triples. The file will then not be a standard cimxml file.
The qualifier decides how uuids will be written in the data. The options are "underscore", "urn" and "namespace".

Tip! If you get a lot of errors saying profile cannot be found, it is because t.graph.metadata_header.profile is None. You can attach a profile manually, or use t.graph.metadata_header.collect_profile to extract it from the header triples. The latter requires the presence of a triple with DCTERMS.conformsTO or MD.Model.profile predicate. The profile decides whether subjects are serialized with rdf:ID or rdf:about, with rdf:about as default when no profile match is found.

In [10]:
output_file_cim = Path.cwd().parent / "fromtrig_tocimxml_grid_eq.xml"
t.graph.serialize(destination=str(output_file_cim), format="cimxml", qualifier="underscore")


<Graph identifier=urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c (<class 'cim_plugin.graph.CIMGraph'>)>

### Serialize to trig
To serialize to trig the header needs to be a part of the graph. If the header has been extracted, use CIMProcess.merge_header to merge it back in.

To get datatypes use CIMProcess.enrich_datatypes. This requires access to a linkML file with the datatype information.

In [11]:
linkmlfile = Path.cwd().parent / "CoreEquipment.linkml.yaml"
g.set_schema(linkmlfile)
g.enrich_literal_datatypes(allow_different_namespaces=True) # You can allow the namespaces in the graph to differ from the model, but the prefix must then be the same
g.merge_header()    # Merge header back in

output_file_trig = Path.cwd().parent / "fromcimxml_totrig_grid_eq.trig"
g.graph.serialize(destination=str(output_file_trig), format="trig")


dc namespace is already mapped to http://purl.org/dc/terms/ - Overriding with mapping to http://purl.org/dc/elements/1.1/


<Graph identifier=urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c (<class 'cim_plugin.graph.CIMGraph'>)>